# 100m Grid → 행정동(Dong) 매핑

## 목적
- 서울 100m grid(cell_id)를 행정동 경계와 매핑
- 각 grid cell이 속한 행정동 코드 / 이름 부여

## 입력
- 100m Grid GeoJSON
- 서울 행정동 SHP (raw)

## 출력
- grid_to_dong.csv

In [1]:
import pandas as pd
import geopandas as gpd

In [2]:
# =========================
# Global Configuration
# =========================

GRID_GEOJSON_PATH = "../data_processed/processed/grid/seoul_grid_100m.geojson"

DONG_SHP_PATH = "../data_processed/raw/admin_boundary/seoul_dong/BND_ADM_DONG_PG.shp"

OUTPUT_PATH = "../data_processed/processed/admin_mapping/grid_to_dong.csv"

TARGET_CRS = "EPSG:3857"

In [3]:
# =========================
# Load 100m Grid
# =========================

grid_gdf = gpd.read_file(GRID_GEOJSON_PATH)

if grid_gdf.crs != TARGET_CRS:
    grid_gdf = grid_gdf.to_crs(TARGET_CRS)

print("Grid rows:", len(grid_gdf))
print("Grid CRS:", grid_gdf.crs)

grid_gdf.head()

Grid rows: 92398
Grid CRS: EPSG:3857


,id,count,cell_id,geometry
0,+141141+45163,1,1411415045163,"POLYGON ((14114100 4516300, 14114200 4516300, ..."
1,+141141+45164,1,1411415045164,"POLYGON ((14114100 4516400, 14114200 4516400, ..."
2,+141141+45165,1,1411415045165,"POLYGON ((14114100 4516500, 14114200 4516500, ..."
3,+141142+45158,1,1411425045158,"POLYGON ((14114200 4515800, 14114300 4515800, ..."
4,+141142+45159,1,1411425045159,"POLYGON ((14114200 4515900, 14114300 4515900, ..."


In [4]:
# =========================
# Load DONG Boundary SHP
# =========================

dong_gdf = gpd.read_file(DONG_SHP_PATH)

if dong_gdf.crs != TARGET_CRS:
    dong_gdf = dong_gdf.to_crs(TARGET_CRS)

print("DONG rows:", len(dong_gdf))
dong_gdf.head()

DONG rows: 3558


,BASE_DATE,ADM_CD,ADM_NM,geometry
0,20240630,24010510,충장동,"POLYGON ((14128587.129 4184723.069, 14128586.0..."
1,20240630,24010540,동명동,"POLYGON ((14129353.724 4184647.703, 14129353.5..."
2,20240630,36680400,안좌면,"MULTIPOLYGON (((14032178.389 4134239.775, 1403..."
3,20240630,36680410,팔금면,"MULTIPOLYGON (((14045532.943 4141812.446, 1404..."
4,20240630,36680420,암태면,"MULTIPOLYGON (((14046119.786 4150695.633, 1404..."


In [5]:
# 서울 지역만 필터링
dong_seoul = dong_gdf[
    dong_gdf["ADM_CD"].astype(str).str.startswith("11")
]

In [6]:
# =========================
# centroid 기반 join + nearest fallback
# =========================

grid_centroid = grid_gdf.copy()
grid_centroid["geometry"] = grid_centroid.geometry.centroid

joined = gpd.sjoin(
    grid_centroid,
    dong_gdf,
    how="left",
    predicate="within"
)

null_mask = joined["ADM_CD"].isna()

if null_mask.sum() > 0:

    fallback = gpd.sjoin_nearest(
        grid_centroid[null_mask],
        dong_gdf,
        how="left",
        distance_col="dist"
    )

    joined.loc[null_mask, ["ADM_CD", "ADM_NM"]] = \
        fallback[["ADM_CD", "ADM_NM"]].values


print("Join completed")
joined.head()

Join completed


,id,count,cell_id,geometry,index_right,BASE_DATE,ADM_CD,ADM_NM
0,+141141+45163,1,1411415045163,POINT (14114150 4516350),1611,20240630,11160690,공항동
1,+141141+45164,1,1411415045164,POINT (14114150 4516450),1611,20240630,11160690,공항동
2,+141141+45165,1,1411415045165,POINT (14114150 4516550),1611,20240630,11160690,공항동
3,+141142+45158,1,1411425045158,POINT (14114250 4515850),1611,20240630,11160690,공항동
4,+141142+45159,1,1411425045159,POINT (14114250 4515950),1611,20240630,11160690,공항동


In [7]:
grid_to_dong = joined[
    ["cell_id", "ADM_CD", "ADM_NM"]
].copy()

print("Null:", grid_to_dong["ADM_CD"].isna().sum())
print("Duplicate:", grid_to_dong["cell_id"].duplicated().sum())

Null: 0
Duplicate: 0


In [9]:
grid_to_dong.to_csv(
    OUTPUT_PATH,
    index=False,
    encoding="utf-8-sig"
)

print("Saved to", OUTPUT_PATH)

Saved to ../data_processed/processed/admin_mapping/grid_to_dong.csv
